In [6]:
from datasets import load_dataset
import re
import random
import torch
from torch.utils.data import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
import evaluate

# Load a smaller subset of data for faster training
dataset = load_dataset("LabHC/bias_in_bios", split='train[:20%]')  # Only 20% of data

def neutralize_text(text: str) -> str:
    """Convert gendered pronouns to neutral forms"""
    replacements = [
        (r'\b([Hh]e|[Ss]he)\s+is\b', lambda m: "They are" if m.group(1)[0].isupper() else "they are"),
        (r'\b([Hh]e|[Ss]he)\s+was\b', lambda m: "They were" if m.group(1)[0].isupper() else "they were"),
        (r'\b([Hh]e|[Ss]he)\s+has\b', lambda m: "They have" if m.group(1)[0].isupper() else "they have"),
        (r'\b([Hh]e|[Ss]he)\b', lambda m: "They" if m.group(1)[0].isupper() else "they"),
        (r'\b([Hh]is|[Hh]er)\b', lambda m: "Their" if m.group(1)[0].isupper() else "their"),
        (r'\b([Hh]im|[Hh]er)\b', lambda m: "Them" if m.group(1)[0].isupper() else "them"),
        (r'\b([Hh]imself|[Hh]erself)\b', lambda m: "Themselves" if m.group(1)[0].isupper() else "themselves")
    ]
    for pattern, repl in replacements:
        text = re.sub(pattern, repl, text)
    return text

# Create training pairs (original, neutralized)
neutral_pairs = []
for i in range(min(5000, len(dataset))):  # Limit to 5000 samples
    original = dataset[i]["hard_text"]
    neutral = neutralize_text(original)
    neutral_pairs.append((original, neutral))

# Shuffle and split
random.shuffle(neutral_pairs)
train_size = int(0.8 * len(neutral_pairs))
train_data = neutral_pairs[:train_size]
val_data = neutral_pairs[train_size:]

# Model config
MODEL_NAME = "t5-small"
TASK_PREFIX = "neutralize: "
MAX_LENGTH = 64  # Reduced sequence length
BATCH_SIZE = 32  # Increased batch size
EPOCHS = 2      # Reduced epochs

class NeutralizationDataset(Dataset):
    def __init__(self, pairs, tokenizer):
        self.pairs = pairs
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        original, neutral = self.pairs[idx]
        inputs = self.tokenizer(
            TASK_PREFIX + original,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        targets = self.tokenizer(
            neutral,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": targets["input_ids"].squeeze(0).masked_fill(
                targets["input_ids"].squeeze(0) == self.tokenizer.pad_token_id, -100
            )
        }

# Initialize model
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# Create datasets
train_dataset = NeutralizationDataset(train_data, tokenizer)
val_dataset = NeutralizationDataset(val_data, tokenizer)

# Optimized training args
training_args = Seq2SeqTrainingArguments(
    output_dir="./neutralizer_model",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_total_limit=1,
    predict_with_generate=True,
    report_to="none"  # Disable logging for speed
)

# Simple metrics
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return rouge.compute(predictions=decoded_preds, references=decoded_labels)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer),
    compute_metrics=compute_metrics
)

print("Training neutralization model...")
trainer.train()

model.save_pretrained("./neutralizer_model")
tokenizer.save_pretrained("./neutralizer_model")

def neutralize_sentence(text):
    inputs = tokenizer(
        TASK_PREFIX + text,
        return_tensors="pt",
        max_length=MAX_LENGTH,
        truncation=True
    )
    outputs = model.generate(
        inputs.input_ids.to(model.device),
        attention_mask=inputs.attention_mask.to(model.device),
        max_length=MAX_LENGTH,
        num_beams=1  
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example usage
test_cases = [
    "He is the project lead of EASy68K.",
    "She earned her degree from Michigan.",
    "The doctor told him that he needs to rest."
]

print("\nNeutralization Examples:")
for case in test_cases:
    print(f"\nOriginal: {case}")
    print(f"Neutral: {neutralize_sentence(case)}")

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-6-5e659d497cd0>:120: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Training neutralization model...


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,0.168181,0.536815,0.519603,0.536967,0.536863
2,0.234900,0.162632,0.536867,0.519721,0.537024,0.536932


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



Neutralization Examples:

Original: He is the project lead of EASy68K.
Neutral: They are the project lead of EASy68K.

Original: She earned her degree from Michigan.
Neutral: They earned their degree from Michigan.

Original: The doctor told him that he needs to rest.
Neutral: The doctor told them that they needs to rest.


In [1]:
from datasets import load_dataset
import re
import random
import torch
from torch.utils.data import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
import evaluate
import numpy as np

dataset = load_dataset("LabHC/bias_in_bios", split='train[:20%]')

def neutralize_text(text: str) -> str:
    """Convert gendered pronouns to neutral forms"""
    replacements = [
        (r'\b([Hh]e|[Ss]he)\s+is\b', lambda m: "They are" if m.group(1)[0].isupper() else "they are"),
        (r'\b([Hh]e|[Ss]he)\s+was\b', lambda m: "They were" if m.group(1)[0].isupper() else "they were"),
        (r'\b([Hh]e|[Ss]he)\s+has\b', lambda m: "They have" if m.group(1)[0].isupper() else "they have"),
        (r'\b([Hh]e|[Ss]he)\b', lambda m: "They" if m.group(1)[0].isupper() else "they"),
        (r'\b([Hh]is|[Hh]er)\b', lambda m: "Their" if m.group(1)[0].isupper() else "their"),
        (r'\b([Hh]im|[Hh]er)\b', lambda m: "Them" if m.group(1)[0].isupper() else "them"),
        (r'\b([Hh]imself|[Hh]erself)\b', lambda m: "Themselves" if m.group(1)[0].isupper() else "themselves")
    ]
    for pattern, repl in replacements:
        text = re.sub(pattern, repl, text)
    return text

neutral_pairs = []
for i in range(min(5000, len(dataset))):
    original = dataset[i]["hard_text"]
    neutral = neutralize_text(original)
    neutral_pairs.append((original, neutral))


random.shuffle(neutral_pairs)
train_size = int(0.8 * len(neutral_pairs))
train_data = neutral_pairs[:train_size]
val_data = neutral_pairs[train_size:]


MODEL_NAME = "t5-small"
TASK_PREFIX = "neutralize: "
MAX_LENGTH = 64
BATCH_SIZE = 32
EPOCHS = 2

class NeutralizationDataset(Dataset):
    def __init__(self, pairs, tokenizer):
        self.pairs = pairs
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        original, neutral = self.pairs[idx]
        inputs = self.tokenizer(
            TASK_PREFIX + original,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        targets = self.tokenizer(
            neutral,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": targets["input_ids"].squeeze(0).masked_fill(
                targets["input_ids"].squeeze(0) == self.tokenizer.pad_token_id, -100
            )
        }

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

train_dataset = NeutralizationDataset(train_data, tokenizer)
val_dataset = NeutralizationDataset(val_data, tokenizer)

training_args = Seq2SeqTrainingArguments(
    output_dir="./neutralizer_model",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_total_limit=1,
    predict_with_generate=True,
    report_to="none" 
)


rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return rouge.compute(predictions=decoded_preds, references=decoded_labels)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer),
    compute_metrics=compute_metrics
)

print("Training neutralization model...")
trainer.train()

model.save_pretrained("./neutralizer_model")
tokenizer.save_pretrained("./neutralizer_model")

def neutralize_sentence(text):
    inputs = tokenizer(
        TASK_PREFIX + text,
        return_tensors="pt",
        max_length=MAX_LENGTH,
        truncation=True
    )
    outputs = model.generate(
        inputs.input_ids.to(model.device),
        attention_mask=inputs.attention_mask.to(model.device),
        max_length=MAX_LENGTH,
        num_beams=1  
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example usage
test_cases = [
    "He is the project lead of EASy68K.",
    "She earned her degree from Michigan.",
    "The doctor told him that he needs to rest."
]

print("\nNeutralization Examples:")
for case in test_cases:
    print(f"\nOriginal: {case}")
    print(f"Neutral: {neutralize_sentence(case)}")

/Users/charanganesh/miniforge3/envs/AI/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/Users/charanganesh/miniforge3/envs/AI/lib/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/var/folde

Training neutralization model...


/Users/charanganesh/miniforge3/envs/AI/lib/python3.10/site-packages/transformers/data/data_collator.py:740: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:281.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss


NameError: name 'np' is not defined